# Дело II-04 · Карта дорогих ошибок

**Бюро аналитических расследований, 10–13 апреля 2026 года.** Поставщик «Компаса» утверждает, что модель оценки жилья «работает в новых районах Калифорнии». Вера Орлова просит проверить не красивое общее число, а именно это обещание.

Вы сравните случайное и групповое по географии разбиения. При случайном соседние кварталы могут оказаться по обе стороны границы; при групповом целые пространственные ячейки остаются невиданными. Это разница между интерполяцией рядом со знакомыми местами и переносом в новый регион.

**Для кого:** студент после дел II-01–II-03. Нужны DataFrame, разбиение на обучающую и тестовую выборки, конвейеры и базовые метрики классификации; регрессия вводится с нуля.

**Результат:** базовая модель, линейная модель и случайный лес; MAE, RMSE и $R^2$; остатки, карта ошибок, ценовые и географические срезы; пятичастная аудиторская записка.


## Маршрут расследования

1. Проверить локальный снимок California Housing и ограничение целевой переменной.
2. Зафиксировать случайную тестовую выборку и географические группы.
3. Сравнить три модели в одинаковых условиях.
4. Перевести метрики из единиц датасета в доллары.
5. Найти структуру в остатках и срезах.
6. Отделить установленный факт от неподтверждённого обещания.

Ориентир времени — 4–5 часов. Внешняя тестовая выборка не используется для выбора гиперпараметров.


In [ ]:
from __future__ import annotations

import hashlib
import random
import urllib.request
import zipfile
from pathlib import Path

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
NOTEBOOK_VARIANT = "solution"
CASE_SLUG = "case-04"
ARCHIVE_NAME = "part-2-case-04.zip"
COURSE_SITE = "https://mkuziuk.github.io/python-tutorial"
IN_COLAB = False

# При локальном запуске используем файлы из каталога дела; в Colab скачиваем архив и проверяем его SHA-256.
try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def find_local_case() -> Path | None:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / f"{CASE_SLUG}.ipynb").is_file()
        ):
            return candidate
        nested = candidate / "projects" / "part-2" / CASE_SLUG
        if (nested / "README.md").exists():
            return nested
    return None

def download_colab_case() -> Path:
    destination = Path("/content") / f"python-tutorial-{CASE_SLUG}"
    destination.mkdir(parents=True, exist_ok=True)
    archive_path = destination / ARCHIVE_NAME
    archive_url = f"{COURSE_SITE}/downloads/{ARCHIVE_NAME}"
    checksum_url = f"{archive_url}.sha256"

    urllib.request.urlretrieve(archive_url, archive_path)
    # Сравниваем SHA-256 архива с опубликованной контрольной суммой перед распаковкой.
    with urllib.request.urlopen(checksum_url) as response:
        expected = response.read().decode("utf-8").split()[0].lower()
    actual = sha256_file(archive_path)
    if actual != expected:
        raise RuntimeError(f"SHA-256 архива не совпал: {actual} != {expected}")

    unpacked = destination / "unpacked"
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpacked)
    matches = sorted(unpacked.rglob(f"{CASE_SLUG}.ipynb"))
    if not matches:
        raise FileNotFoundError(f"В архиве нет {CASE_SLUG}.ipynb")
    return matches[0].parent

# DATA_DIR и ARTIFACTS_DIR строятся от найденного каталога дела, поэтому текущая папка не влияет на пути.
CASE_DIR = find_local_case()
if CASE_DIR is None and IN_COLAB:
    CASE_DIR = download_colab_case()
if CASE_DIR is None:
    raise FileNotFoundError(
        f"Не найден каталог {CASE_SLUG}. Запустите тетрадь из каталога дела."
    )

DATA_DIR = CASE_DIR / "data"
ARTIFACTS_DIR = CASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
print(f"Среда: {'Colab' if IN_COLAB else 'local'} | дело: {CASE_DIR}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Карточка источника и контроль целостности

Бюро поставляет фиксированный CSV: 20 640 блок-групп переписи 1990 года, восемь числовых признаков и `MedHouseVal`. Целевая переменная измеряется в сотнях тысяч долларов. Файл получен через `fetch_california_housing` и больше не зависит от сети.

`SOURCE.md`, `LICENSE.txt` и `dataset_manifest.json` фиксируют происхождение, версию конвертации и SHA-256. Сначала проверяем байты, только потом строим выводы.


In [ ]:
manifest = json.loads((DATA_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
data_path = DATA_DIR / manifest["filename"]
actual_digest = sha256_file(data_path)
assert actual_digest == manifest["sha256"], "Снимок данных изменился"

housing = pd.read_csv(data_path)
assert housing.shape == (20_640, 9)
display(housing.head())
print(f"Строк: {len(housing):,}; пропусков: {int(housing.isna().sum().sum())}")


In [ ]:
audit = pd.DataFrame({
    "dtype": housing.dtypes.astype(str),
    "missing": housing.isna().sum(),
    "minimum": housing.min(numeric_only=True),
    "median": housing.median(numeric_only=True),
    "maximum": housing.max(numeric_only=True),
})
display(audit.round(3))

target_cap = float(manifest["target_cap"])
capped_count = int(np.isclose(housing["MedHouseVal"], target_cap).sum())
print(f"Строк на верхней границе target {target_cap}: {capped_count}")


Верхняя граница — не просто неудобный выброс: значения выше примерно 500 000 долларов были сведены к одному числу. Поэтому модель не может научиться различать дорогие объекты за этой границей, а ошибки в верхнем ценовом срезе искусственно искажены. Это ограничение данных, не свойство алгоритма.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(housing, x="MedHouseVal", bins=40, ax=axes[0], color="#3157a4")
axes[0].axvline(target_cap, color="#9b4b3f", linestyle="--", label="target cap")
axes[0].set(title="Распределение target", xlabel="Цена, × $100 000")
axes[0].legend()
scatter = axes[1].scatter(
    housing["Longitude"], housing["Latitude"],
    c=housing["MedHouseVal"], s=4, alpha=0.45, cmap="viridis"
)
axes[1].set(title="Цена имеет пространственную структуру", xlabel="Longitude", ylabel="Latitude")
fig.colorbar(scatter, ax=axes[1], label="MedHouseVal")
plt.tight_layout()
plt.show()


## 2. Постановка и две границы проверки

Мы предсказываем непрерывную величину. Для ошибки $e_i=y_i-\hat y_i$:

- $MAE=\frac{1}{n}\sum|e_i|$ — типичная абсолютная ошибка;
- $RMSE=\sqrt{\frac{1}{n}\sum e_i^2}$ — сильнее штрафует большие промахи;
- $R^2=1-\frac{\sum e_i^2}{\sum(y_i-\bar y)^2}$ — доля вариации относительно базового прогноза среднего.

MAE и RMSE здесь умножаем на 100 000, чтобы вернуть доллары. $R^2$ не имеет единицы и может быть отрицательным.


In [ ]:
target_column = "MedHouseVal"
X = housing.drop(columns=target_column)
y = housing[target_column]
all_positions = np.arange(len(housing))

# Случайное разбиение проверяет интерполяцию рядом со знакомыми регионами.
random_train, random_test = train_test_split(
    all_positions, test_size=0.20, random_state=RANDOM_STATE
)
print(f"Случайный split: train={len(random_train):,}, тест={len(random_test):,}")


### Упражнение: единица независимости — регион

Создайте ячейку примерно 0,5° × 0,5° из longitude и latitude. Затем `GroupShuffleSplit` должен удержать все строки одной ячейки только с одной стороны границы. Проверьте пересечение множеств групп явно.


In [ ]:
# BEGIN SOLUTION
# Объединяем координаты в ячейки 0,5° × 0,5° и используем идентификатор ячейки как группу.
longitude_bin = np.floor((X["Longitude"] + 125.0) * 2).astype(int)
latitude_bin = np.floor((X["Latitude"] - 32.0) * 2).astype(int)
region_groups = longitude_bin.astype(str) + "_" + latitude_bin.astype(str)
# END SOLUTION

region_splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=RANDOM_STATE
)
grouped_train, grouped_test = next(
    region_splitter.split(X, y, groups=region_groups)
)
# group_overlap должен быть пустым перед оценкой модели.
group_overlap = set(region_groups.iloc[grouped_train]) & set(region_groups.iloc[grouped_test])
print(
    f"Групп: {region_groups.nunique()}, train={len(grouped_train):,}, "
    f"test={len(grouped_test):,}, пересечение={len(group_overlap)}"
)


## 3. Три уровня сложности

`DummyRegressor` всегда предсказывает среднее обучающей выборки и задаёт нижнюю планку. Линейная регрессия проверяет простую глобальную зависимость. Случайный лес улавливает нелинейности, но особенно легко пользуется похожими соседними строками — поэтому смена разбиения важна.

**Упражнение:** клонируйте и обучите каждую модель только на обучающей выборке; верните таблицу метрик, предсказания и обученные объекты.


In [ ]:
model_suite = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "linear": make_pipeline(StandardScaler(), LinearRegression()),
    "random_forest": RandomForestRegressor(
        n_estimators=120,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

# BEGIN SOLUTION
# Один набор моделей оцениваем на случайном и географическом разбиениях.
def evaluate_suite(train_positions, test_positions):
    rows, predictions, fitted_models = [], {}, {}
    for model_name, template in model_suite.items():
        fitted = clone(template).fit(X.iloc[train_positions], y.iloc[train_positions])
        predicted = fitted.predict(X.iloc[test_positions])
        rows.append({
            "model": model_name,
            "mae": mean_absolute_error(y.iloc[test_positions], predicted),
            "rmse": root_mean_squared_error(y.iloc[test_positions], predicted),
            "r2": r2_score(y.iloc[test_positions], predicted),
        })
        predictions[model_name] = predicted
        fitted_models[model_name] = fitted
    return pd.DataFrame(rows).set_index("model"), predictions, fitted_models
# END SOLUTION

random_results, random_predictions, random_fitted = evaluate_suite(
    random_train, random_test
)
# MAE и RMSE измеряются в сотнях тысяч долларов; умножение на 100 000 переводит их в доллары.
display(random_results.assign(
    mae_usd=random_results["mae"] * 100_000,
    rmse_usd=random_results["rmse"] * 100_000,
).round(3))


In [ ]:
grouped_results, grouped_predictions, grouped_fitted = evaluate_suite(
    grouped_train, grouped_test
)
comparison = pd.concat(
    {"random_holdout": random_results, "region_holdout": grouped_results},
    names=["split", "model"],
)
display(comparison.assign(
    mae_usd=comparison["mae"] * 100_000,
    rmse_usd=comparison["rmse"] * 100_000,
).round(3))


## 4. Остатки и карта промахов

Общая метрика отвечает «сколько в среднем», но не «где» и не «для каких цен». Остаток определяем как `y_true - y_pred`: положительный означает недооценку, отрицательный — переоценку.


In [ ]:
if "random_forest" in random_predictions:
    random_residuals = y.iloc[random_test].to_numpy() - random_predictions["random_forest"]
    grouped_residuals = y.iloc[grouped_test].to_numpy() - grouped_predictions["random_forest"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    axes[0].scatter(random_predictions["random_forest"], random_residuals, s=7, alpha=0.3)
    axes[1].scatter(grouped_predictions["random_forest"], grouped_residuals, s=7, alpha=0.3)
    for ax, title in zip(axes, ["Случайная тестовая выборка", "Новые региональные ячейки"], strict=True):
        ax.axhline(0, color="black", linewidth=1)
        ax.set(title=title, xlabel="Предсказание, × $100 000", ylabel="Остаток")
    plt.tight_layout()
    plt.show()
else:
    print("Добавьте random_forest в упражнении, чтобы увидеть остатки.")


In [ ]:
if "random_forest" in grouped_predictions:
    map_frame = X.iloc[grouped_test][["Longitude", "Latitude"]].copy()
    map_frame["absolute_error_usd"] = np.abs(grouped_residuals) * 100_000
    fig, ax = plt.subplots(figsize=(7, 6))
    points = ax.scatter(
        map_frame["Longitude"], map_frame["Latitude"],
        c=map_frame["absolute_error_usd"], s=10, alpha=0.65, cmap="magma"
    )
    fig.colorbar(points, ax=ax, label="Абсолютная ошибка, USD")
    ax.set(title="Карта ошибок на невиданных региональных ячейках", xlabel="Longitude", ylabel="Latitude")
    plt.show()
else:
    map_frame = pd.DataFrame()
    print("Карта появится после реализации random_forest.")


### Упражнение: агрегат скрывает срезы

Сравните MAE по квартилям истинной цены и по северной/южной части групповой тестовой выборки. Обязательно показывайте `n`: маленький срез даёт нестабильную оценку.


In [ ]:
if "random_forest" in grouped_predictions:
    slice_frame = X.iloc[grouped_test][["Latitude", "Longitude"]].copy()
    slice_frame["actual"] = y.iloc[grouped_test].to_numpy()
    slice_frame["prediction"] = grouped_predictions["random_forest"]
    slice_frame["absolute_error_usd"] = np.abs(
        slice_frame["actual"] - slice_frame["prediction"]
    ) * 100_000
    # BEGIN SOLUTION
    slice_frame["price_quartile"] = pd.qcut(
        slice_frame["actual"], 4, labels=["Q1", "Q2", "Q3", "Q4"]
    )
    slice_frame["geography"] = np.where(
        slice_frame["Latitude"] >= 37.0, "north", "south"
    )
    price_slices = slice_frame.groupby("price_quartile", observed=True).agg(
        n=("actual", "size"), mae_usd=("absolute_error_usd", "mean")
    )
    geography_slices = slice_frame.groupby("geography").agg(
        n=("actual", "size"), mae_usd=("absolute_error_usd", "mean")
    )
    # END SOLUTION
    display(price_slices.round(0))
    display(geography_slices.round(0))
else:
    price_slices = geography_slices = pd.DataFrame()
    print("Срезы ждут предсказаний random_forest.")


## 5. Аудиторская записка

Записка обязана разделять пять вещей: установленный факт, поддержанную интерпретацию, что не доказано, ограничения и действие. Нельзя превращать пространственную корреляцию в причинное объяснение цены.


In [ ]:
# BEGIN SOLUTION
memo = {
    "established_fact": (
        "На фиксированном снимке random forest превосходит baseline, но его RMSE "
        "на региональном holdout заметно выше, чем на случайном."
    ),
    "supported_interpretation": (
        "Случайный split измеряет главным образом интерполяцию среди географически "
        "похожих кварталов и завышает поддержку заявления о новых регионах."
    ),
    "not_proven": (
        "Анализ не доказывает причин цены, пригодности для текущего рынка или "
        "умышленного введения в заблуждение конкретным сотрудником."
    ),
    "limitations": (
        "Перепись 1990 года устарела; target top-coded; один групповой split и грубые "
        "ячейки не описывают все варианты географического переноса."
    ),
    "recommended_action": (
        "Отклонить формулировку «работает в новых районах» до независимой проверки "
        "на заранее определённых регионах и публиковать карты и срезы ошибок."
    ),
}
# END SOLUTION

memo_text = "# Аудиторская записка II-04\n\n" + "\n\n".join(
    f"## {key}\n{value}" for key, value in memo.items()
) + "\n"
(ARTIFACTS_DIR / "audit_memo.md").write_text(memo_text, encoding="utf-8")
print(memo_text)


In [ ]:
if NOTEBOOK_VARIANT == "solution":
    assert not group_overlap
    assert random_results.loc["random_forest", "rmse"] < random_results.loc["dummy_mean", "rmse"]
    assert grouped_results.loc["random_forest", "rmse"] > random_results.loc["random_forest", "rmse"] + 0.08
    assert capped_count > 900
    assert set(memo) == {
        "established_fact", "supported_interpretation", "not_proven",
        "limitations", "recommended_action"
    }
    print("Проверки решения II-04 пройдены.")
else:
    print("Учебный вариант: строгие проверки включатся после сверки с решением.")


## Дело закрыто

Перезапустите ядро и выполните **Run All**. Сверьте форму результата с `check_result.md`: целостность снимка, нулевое пересечение регионов, базовую модель, метрики в долларах, остатки, карта, срезы с `n` и пятичастная записка.

**Типичная ошибка:** увидеть худший групповой результат и начать подбирать размер ячейки по тестовой выборке. Границу проверки определяют до оценки; иначе новая тестовая выборка тоже становится частью обучения.

**Расширение:** повторите групповое разбиение для нескольких заранее объявленных начальных значений генератора и покажите диапазон метрик, не меняя модель по результатам тестовой выборки.
